<a href="https://colab.research.google.com/github/7t8bb58bvk-cloud/pokechamp-ai/blob/main/31%E6%97%A54%E6%99%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

print(uploaded.keys())

Saving pokechamp-ai-handoff.zip to pokechamp-ai-handoff.zip
dict_keys(['pokechamp-ai-handoff.zip'])


In [ ]:
import engine.damage_core_v2 as dc

if not hasattr(dc, "calculate_damage"):
    dc.calculate_damage = dc.damage_rolls

print("patched:", hasattr(dc, "calculate_damage"))

patched: True


In [ ]:
zip_path="/content/pokechamp-ai-handoff.zip"

In [ ]:
from zipfile import ZipFile

with ZipFile(zip_path,"r") as z:
    z.extractall("/content")

print("展開完了")

展開完了


In [ ]:
import os

print("現在:", os.getcwd())
print("\n/content の中身")
print(os.listdir("/content"))

現在: /content

/content の中身
['.config', 'utils', 'tests', 'data', 'HANDOVER.md', 'learning', '.git', 'PROJECT_STATUS.md', 'engine', '_handoff_snapshots', '暫定バックアップ.ipynb', 'TODO.md', 'ROADMAP.md', 'README.md', 'Untitled7.ipynb', 'KNOWN_ISSUES.md', 'docs', 'pokechamp-ai-handoff.zip', 'AI_DEVELOPMENT_RULES.md', 'ARCHITECTURE.md', 'ai', 'battle', 'sample_data']


In [ ]:
%cd /content
!git status

/content
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   battle/__pycache__/__init__.cpython-312.pyc
	modified:   battle/__pycache__/action.cpython-312.pyc
	modified:   battle/__pycache__/battle_state.cpython-312.pyc
	modified:   battle/__pycache__/field.cpython-312.pyc
	modified:   battle/__pycache__/pokemon.cpython-312.pyc
	modified:   battle/__pycache__/side.cpython-312.pyc
	modified:   battle/__pycache__/team_builder.cpython-312.pyc
	modified:   data/__pycache__/__init__.cpython-312.pyc
	modified:   data/__pycache__/move_database.cpython-312.pyc
	modified:   data/__pycache__/pokemon_database.cpython-312.pyc
	modified:   data/__pycache__/type_chart.cpython-312.pyc
	modified:   engine/__pycache__/__init__.cpython-312.pyc
	modified:   engine/__pycache__/action_value_v1.cpython-312.pyc
	modified:  

In [ ]:
import engine.damage_core_v2 as dc

print(dir(dc))

['DamageProfileV2', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_accuracy', '_burn_modifier', '_item_modifier', '_raw_damage', '_stab', '_type_effectiveness', '_weather_modifier', 'annotations', 'calculate_damage', 'damage_rolls', 'dataclass', 'floor', 'get_modified_stat', 'get_move', 'type_multiplier']


In [ ]:
from engine.damage_core_v2 import calculate_damage

...

def xxxx():
    ...

In [ ]:
# ===== システム健全性チェック =====

import importlib
import traceback

modules = [
    "engine.damage_core_v2",
    "engine.stat_core_v2",
    "engine.turn_core_v1",
    "engine.switch_value_v1",
    "engine.action_value_v1",
    "engine.decision_manager_v1",
    "battle.battle_state",
    "battle.action",
    "battle.pokemon",
    "battle.side",
]

print("=" * 60)

for m in modules:
    try:
        importlib.import_module(m)
        print(f"✅ {m}")
    except Exception:
        print(f"❌ {m}")
        traceback.print_exc(limit=2)

print("=" * 60)
print("チェック終了")

✅ engine.damage_core_v2
✅ engine.stat_core_v2
✅ engine.turn_core_v1
✅ engine.switch_value_v1
✅ engine.action_value_v1
✅ engine.decision_manager_v1
✅ battle.battle_state
✅ battle.action
✅ battle.pokemon
✅ battle.side
チェック終了


In [ ]:
from engine.decision_manager_v1 import DecisionManagerV1

dm = DecisionManagerV1()

print("DecisionManager OK")

DecisionManager OK


In [ ]:
from engine.decision_manager_v1 import score_action, ACTION_MOVE, ACTION_SWITCH

class MockSide:
    def __init__(self, active_idx=0):
        self.active_index = active_idx
        self.prev_action = None
        self.team = [type("P", (), {"fainted": False})() for _ in range(3)]

class MockPokemon:
    def __init__(self, hp):
        self.current_hp = hp
        self.moves = []

class MockMove:
    def __init__(self, base_power):
        self.base_power = base_power

class MockAction:
    def __init__(self, action_type, move=None, switch_idx=None):
        self.action_type = action_type
        self.move = move
        self.switch_index = switch_idx

player = MockPokemon(100)
opponent = MockPokemon(120)
player.moves = [MockMove(80), MockMove(10)]

state = type(
    "State",
    (),
    {
        "player": player,
        "opponent": opponent,
        "player_side": MockSide(0),
        "opponent_side": MockSide(0),
        "field": None,
        "turn": 1,
    },
)

action_move = MockAction(ACTION_MOVE, move=player.moves[0])
action_switch = MockAction(ACTION_SWITCH, switch_idx=0)

print("move   =", score_action(state, action_move, "player"))
print("switch =", score_action(state, action_switch, "player"))

move   = -300000.0
switch = -500000.0


In [ ]:
import inspect
import engine.decision_manager_v1 as dm

print(inspect.getsource(dm)[:6000])


from __future__ import annotations

from battle.action import ACTION_MOVE, ACTION_SWITCH
from engine.damage_core_v2 import damage_rolls

try:
    from engine.stat_core_v2 import calculate_speed
except Exception:
    try:
        from engine.stat_engine import calculate_speed
    except Exception:
        def calculate_speed(pokemon, field, side_name="player"):
            return float(getattr(pokemon, "speed", 0) or 0)


def _alive_count(side) -> int:
    try:
        return int(side.alive_count())
    except Exception:
        return sum(
            1 for p in getattr(side, "team", []) or []
            if not getattr(p, "fainted", False)
        )


def _priority(action) -> int:
    if action is None:
        return 0
    kind = getattr(action, "action_type", None)
    if kind == ACTION_SWITCH:
        return 6
    if kind == ACTION_MOVE and str(getattr(action, "move", "")).lower() == "protect":
        return 4
    return int(getattr(action, "priority", 0) or 0)


def _our_goes_fi

In [ ]:
import inspect
import engine.decision_manager_v1 as dm

src = inspect.getsource(dm)

print("文字数:", len(src))
print(src[-2000:])

文字数: 33707
= ACTION_SWITCH]

    if opening:
        safe_moves = [a for a in move_actions if _safe_opening_move_v60(state, me, opp, a)]
        emergency_switches = [a for a in switch_actions if _emergency_switch_v60(state, me, opp, a, side_name)]

        # STRONGER OPENING RULE:
        # If there is at least one move, opening turn prefers move actions.
        # Only when there are no usable moves, emergency switches can compete.
        if safe_moves:
            actions = safe_moves
        elif move_actions:
            actions = move_actions
        elif emergency_switches:
            actions = emergency_switches
        else:
            actions = actions

    else:
        hp = _hp_ratio_v60(me)
        emergency_switches = [a for a in switch_actions if _emergency_switch_v60(state, me, opp, a, side_name)]

        # Prevent switch loops when HP is still healthy.
        if hp > 0.55:
            actions = move_actions
        elif hp > 0.35:
            actions = move_action

In [ ]:
# v61 bootstrap: current v60 code stays untouched

import engine.decision_manager_v1 as dm

# 1) calculate_damage の互換は維持
import engine.damage_core_v2 as dc
if not hasattr(dc, "calculate_damage"):
    dc.calculate_damage = dc.damage_rolls

# 2) v60 の選択ロジックが残っているか確認
print("DecisionManagerV1:", hasattr(dm, "DecisionManagerV1"))
print("rank_actions_v60:", hasattr(dm, "rank_actions_v60"))
print("choose_best_v60:", hasattr(dm, "choose_best_v60"))
print("calculate_damage patched:", hasattr(dc, "calculate_damage"))

# 3) v61 用の入口だけ先に作る
if not hasattr(dm, "rank_actions_v61"):
    dm.rank_actions_v61 = dm.rank_actions_v60
if not hasattr(dm, "choose_best_v61"):
    dm.choose_best_v61 = dm.choose_best_v60

print("v61 aliases ready")

DecisionManagerV1: True
rank_actions_v60: True
choose_best_v60: True
calculate_damage patched: True
v61 aliases ready


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

from battle.action import ACTION_MOVE, ACTION_SWITCH
from engine.damage_core_v2 import damage_rolls

try:
    from engine.stat_core_v2 import calculate_speed
except Exception:
    try:
        from engine.stat_engine import calculate_speed
    except Exception:
        def calculate_speed(pokemon, field, side_name="player"):
            return float(getattr(pokemon, "speed", 0) or 0)


def _hp_ratio(pokemon) -> float:
    max_hp = float(getattr(pokemon, "max_hp", 1) or 1)
    cur_hp = float(getattr(pokemon, "current_hp", 0) or 0)
    return max(0.0, min(1.0, cur_hp / max_hp))


def _alive_count(side) -> int:
    try:
        return int(side.alive_count())
    except Exception:
        return sum(
            1 for p in (getattr(side, "team", []) or [])
            if not getattr(p, "fainted", False)
        )


def _profile_damage(attacker, defender, move, field):
    prof = damage_rolls(attacker, defender, move, field)

    if isinstance(prof, tuple):
        if len(prof) >= 2:
            raw = float(prof[0] or 0.0)
            ko = float(prof[1] or 0.0)
        else:
            raw = float(prof[0] or 0.0)
            ko = 0.0
    else:
        raw = float(getattr(prof, "avg_damage", getattr(prof, "raw_damage", 0.0)) or 0.0)
        ko = float(getattr(prof, "ko_chance", getattr(prof, "ko_prob", 0.0)) or 0.0)

    return raw, ko


def _best_move_stats(state, me, opp):
    best_raw = 0.0
    best_ko = 0.0
    best_move = None

    for mv in getattr(me, "moves", []) or []:
        if not mv:
            continue
        try:
            raw, ko = _profile_damage(me, opp, mv, state.field)
            if raw > best_raw:
                best_raw = raw
                best_ko = ko
                best_move = mv
        except Exception:
            pass

    return best_move, float(best_raw), float(best_ko)


def _opp_threat_into(state, defender, attacker) -> float:
    threat = 0.0
    for mv in getattr(attacker, "moves", []) or []:
        if not mv:
            continue
        try:
            raw, ko = _profile_damage(attacker, defender, mv, state.field)
            threat = max(threat, raw + ko * 2500.0)
        except Exception:
            pass
    return float(threat)


def _entry_hazard_penalty(side) -> float:
    if side is None:
        return 0.0

    penalty = 0.0
    spikes = int(getattr(side, "spikes", 0) or 0)
    toxic_spikes = int(getattr(side, "toxic_spikes", 0) or 0)
    stealth_rock = bool(getattr(side, "stealth_rock", False))
    sticky_web = bool(getattr(side, "sticky_web", False))

    penalty += spikes * 160.0
    penalty += toxic_spikes * 120.0
    penalty += 220.0 if stealth_rock else 0.0
    penalty += 90.0 if sticky_web else 0.0

    return float(penalty)


def _field_tempo_bonus(state, me, opp, side_name="player") -> float:
    me_speed = calculate_speed(me, state.field, side_name)
    opp_speed = calculate_speed(
        opp,
        state.field,
        "opponent" if side_name == "player" else "player",
    )

    if me_speed >= opp_speed:
        return 180.0 + min(220.0, (me_speed - opp_speed) * 2.0)
    return -160.0 - min(220.0, (opp_speed - me_speed) * 1.8)


@dataclass
class BoardScoreBreakdown:
    own_hp: float = 0.0
    opp_hp: float = 0.0
    damage: float = 0.0
    ko: float = 0.0
    threat: float = 0.0
    tempo: float = 0.0
    hazards: float = 0.0
    alive: float = 0.0
    endgame: float = 0.0
    total: float = 0.0


class BoardValueV1:
    """
    盤面評価:
    - 今の対面が有利か
    - 速度で先手を取れるか
    - 相手の圧が高いか
    - 交代負荷やステルスロック等の負担があるか
    """

    OWN_HP_WEIGHT = 2600.0
    OPP_HP_WEIGHT = 3000.0
    DAMAGE_WEIGHT = 8.0
    KO_WEIGHT = 2200.0
    THREAT_WEIGHT = 8.0
    TEMPO_WEIGHT = 1.0
    HAZARD_WEIGHT = 1.0
    ALIVE_WEIGHT = 120.0
    ENDGAME_WEIGHT = 300.0

    @staticmethod
    def evaluate(state, me, opp, side_name="player", action=None) -> float:
        my_hp = _hp_ratio(me)
        opp_hp = _hp_ratio(opp)

        best_move, best_raw, best_ko = _best_move_stats(state, me, opp)
        opp_threat = _opp_threat_into(state, me, opp)
        tempo = _field_tempo_bonus(state, me, opp, side_name)

        own_side = state.player_side if side_name == "player" else state.opponent_side
        opp_side = state.opponent_side if side_name == "player" else state.player_side

        hazards = -_entry_hazard_penalty(own_side) + 0.5 * _entry_hazard_penalty(opp_side)

        alive_diff = _alive_count(own_side) - _alive_count(opp_side)

        score = 0.0
        score += my_hp * BoardValueV1.OWN_HP_WEIGHT
        score -= opp_hp * BoardValueV1.OPP_HP_WEIGHT
        score += best_raw * BoardValueV1.DAMAGE_WEIGHT
        score += best_ko * BoardValueV1.KO_WEIGHT
        score -= opp_threat * BoardValueV1.THREAT_WEIGHT
        score += tempo * BoardValueV1.TEMPO_WEIGHT
        score += hazards * BoardValueV1.HAZARD_WEIGHT
        score += alive_diff * BoardValueV1.ALIVE_WEIGHT

        # 終盤: 生存体数が少ないほどHPの価値を少し上げる
        if _alive_count(own_side) <= 2 or _alive_count(opp_side) <= 2:
            score += (my_hp - opp_hp) * BoardValueV1.ENDGAME_WEIGHT

        # 行動候補があるなら、攻撃/交代の質も少しだけ反映
        if action is not None:
            kind = getattr(action, "action_type", None)
            if kind == ACTION_MOVE and best_move is not None:
                score += 140.0
            elif kind == ACTION_SWITCH:
                score -= 60.0

        return float(score)

    @staticmethod
    def breakdown(state, me, opp, side_name="player", action=None) -> BoardScoreBreakdown:
        my_hp = _hp_ratio(me)
        opp_hp = _hp_ratio(opp)
        best_move, best_raw, best_ko = _best_move_stats(state, me, opp)
        opp_threat = _opp_threat_into(state, me, opp)
        tempo = _field_tempo_bonus(state, me, opp, side_name)

        own_side = state.player_side if side_name == "player" else state.opponent_side
        opp_side = state.opponent_side if side_name == "player" else state.player_side
        hazards = -_entry_hazard_penalty(own_side) + 0.5 * _entry_hazard_penalty(opp_side)
        alive_diff = _alive_count(own_side) - _alive_count(opp_side)

        endgame = 0.0
        if _alive_count(own_side) <= 2 or _alive_count(opp_side) <= 2:
            endgame = (my_hp - opp_hp) * BoardValueV1.ENDGAME_WEIGHT

        total = (
            my_hp * BoardValueV1.OWN_HP_WEIGHT
            - opp_hp * BoardValueV1.OPP_HP_WEIGHT
            + best_raw * BoardValueV1.DAMAGE_WEIGHT
            + best_ko * BoardValueV1.KO_WEIGHT
            - opp_threat * BoardValueV1.THREAT_WEIGHT
            + tempo * BoardValueV1.TEMPO_WEIGHT
            + hazards * BoardValueV1.HAZARD_WEIGHT
            + alive_diff * BoardValueV1.ALIVE_WEIGHT
            + endgame
        )

        return BoardScoreBreakdown(
            own_hp=my_hp,
            opp_hp=opp_hp,
            damage=best_raw,
            ko=best_ko,
            threat=opp_threat,
            tempo=tempo,
            hazards=hazards,
            alive=alive_diff,
            endgame=endgame,
            total=float(total),
        )


def board_value(state, me, opp, side_name="player", action=None) -> float:
    return BoardValueV1.evaluate(state, me, opp, side_name=side_name, action=action)


def board_breakdown(state, me, opp, side_name="player", action=None) -> BoardScoreBreakdown:
    return BoardValueV1.breakdown(state, me, opp, side_name=side_name, action=action)

In [ ]:
import os

for root, dirs, files in os.walk("/content/engine"):
    for f in sorted(files):
        print(f)

__init__.py
action_value_v1.py
advanced_evaluation.py
damage_core_v2.py
damage_engine.py
decision_manager_v1.py
decision_manager_v1.py.bak_before_v32
decision_manager_v1.py.bak_before_v33
decision_manager_v1.py.bak_before_v34
decision_manager_v1.py.bak_before_v35
decision_manager_v1.py.bak_before_v36
decision_manager_v1.py.bak_before_v37
decision_manager_v1.py.bak_before_v38
decision_manager_v1.py.bak_before_v39
decision_manager_v1.py.bak_before_v40
decision_manager_v1.py.bak_before_v41
decision_manager_v1.py.bak_before_v42
decision_manager_v1.py.bak_before_v43
decision_manager_v1.py.bak_before_v44
decision_manager_v1.py.bak_before_v44_fix
decision_manager_v1.py.bak_before_v45
decision_manager_v1.py.bak_before_v45_filter
decision_manager_v1.py.bak_before_v50_clean
decision_manager_v1.py.bak_before_v51
decision_manager_v1.py.bak_before_v52
decision_manager_v1.py.bak_before_v53
decision_manager_v1.py.bak_before_v54
decision_manager_v1.py.bak_before_v55
decision_manager_v1.py.bak_before_v

In [ ]:
from pathlib import Path
from textwrap import dedent

file_path = Path("/content/engine/board_value_v1.py")

content = dedent(r'''
from __future__ import annotations

from dataclasses import dataclass

from battle.action import ACTION_MOVE, ACTION_SWITCH
from engine.damage_core_v2 import damage_rolls

try:
    from engine.stat_core_v2 import calculate_speed
except Exception:
    try:
        from engine.stat_engine import calculate_speed
    except Exception:
        def calculate_speed(pokemon, field, side_name="player"):
            return float(getattr(pokemon, "speed", 0) or 0)


def _hp_ratio(pokemon) -> float:
    max_hp = float(getattr(pokemon, "max_hp", 1) or 1)
    cur_hp = float(getattr(pokemon, "current_hp", 0) or 0)
    return max(0.0, min(1.0, cur_hp / max_hp))


def _alive_count(side) -> int:
    try:
        return int(side.alive_count())
    except Exception:
        return sum(
            1 for p in (getattr(side, "team", []) or [])
            if not getattr(p, "fainted", False)
        )


def _profile_damage(attacker, defender, move, field):
    prof = damage_rolls(attacker, defender, move, field)

    if isinstance(prof, tuple):
        raw = float(prof[0] or 0.0) if len(prof) >= 1 else 0.0
        ko = float(prof[1] or 0.0) if len(prof) >= 2 else 0.0
        return raw, ko

    raw = float(getattr(prof, "avg_damage", getattr(prof, "raw_damage", 0.0)) or 0.0)
    ko = float(getattr(prof, "ko_chance", getattr(prof, "ko_prob", 0.0)) or 0.0)
    return raw, ko


def _best_move_stats(state, me, opp):
    best_move = None
    best_raw = 0.0
    best_ko = 0.0

    for mv in getattr(me, "moves", []) or []:
        if not mv:
            continue
        try:
            raw, ko = _profile_damage(me, opp, mv, state.field)
            if raw > best_raw:
                best_raw = raw
                best_ko = ko
                best_move = mv
        except Exception:
            pass

    return best_move, float(best_raw), float(best_ko)


def _opp_threat_into(state, defender, attacker) -> float:
    threat = 0.0
    for mv in getattr(attacker, "moves", []) or []:
        if not mv:
            continue
        try:
            raw, ko = _profile_damage(attacker, defender, mv, state.field)
            threat = max(threat, raw + ko * 2500.0)
        except Exception:
            pass
    return float(threat)


def _entry_hazard_penalty(side) -> float:
    if side is None:
        return 0.0

    penalty = 0.0
    spikes = int(getattr(side, "spikes", 0) or 0)
    toxic_spikes = int(getattr(side, "toxic_spikes", 0) or 0)
    stealth_rock = bool(getattr(side, "stealth_rock", False))
    sticky_web = bool(getattr(side, "sticky_web", False))

    penalty += spikes * 160.0
    penalty += toxic_spikes * 120.0
    penalty += 220.0 if stealth_rock else 0.0
    penalty += 90.0 if sticky_web else 0.0
    return float(penalty)


def _field_tempo_bonus(state, me, opp, side_name="player") -> float:
    me_speed = calculate_speed(me, state.field, side_name)
    opp_speed = calculate_speed(
        opp,
        state.field,
        "opponent" if side_name == "player" else "player",
    )

    if me_speed >= opp_speed:
        return 180.0 + min(220.0, (me_speed - opp_speed) * 2.0)
    return -160.0 - min(220.0, (opp_speed - me_speed) * 1.8)


@dataclass
class BoardScoreBreakdown:
    own_hp: float = 0.0
    opp_hp: float = 0.0
    damage: float = 0.0
    ko: float = 0.0
    threat: float = 0.0
    tempo: float = 0.0
    hazards: float = 0.0
    alive: float = 0.0
    endgame: float = 0.0
    total: float = 0.0


class BoardValueV1:
    OWN_HP_WEIGHT = 2600.0
    OPP_HP_WEIGHT = 3000.0
    DAMAGE_WEIGHT = 8.0
    KO_WEIGHT = 2200.0
    THREAT_WEIGHT = 8.0
    TEMPO_WEIGHT = 1.0
    HAZARD_WEIGHT = 1.0
    ALIVE_WEIGHT = 120.0
    ENDGAME_WEIGHT = 300.0

    @staticmethod
    def evaluate(state, me, opp, side_name="player", action=None) -> float:
        my_hp = _hp_ratio(me)
        opp_hp = _hp_ratio(opp)

        best_move, best_raw, best_ko = _best_move_stats(state, me, opp)
        opp_threat = _opp_threat_into(state, me, opp)
        tempo = _field_tempo_bonus(state, me, opp, side_name)

        own_side = state.player_side if side_name == "player" else state.opponent_side
        opp_side = state.opponent_side if side_name == "player" else state.player_side

        hazards = -_entry_hazard_penalty(own_side) + 0.5 * _entry_hazard_penalty(opp_side)
        alive_diff = _alive_count(own_side) - _alive_count(opp_side)

        score = 0.0
        score += my_hp * BoardValueV1.OWN_HP_WEIGHT
        score -= opp_hp * BoardValueV1.OPP_HP_WEIGHT
        score += best_raw * BoardValueV1.DAMAGE_WEIGHT
        score += best_ko * BoardValueV1.KO_WEIGHT
        score -= opp_threat * BoardValueV1.THREAT_WEIGHT
        score += tempo * BoardValueV1.TEMPO_WEIGHT
        score += hazards * BoardValueV1.HAZARD_WEIGHT
        score += alive_diff * BoardValueV1.ALIVE_WEIGHT

        if _alive_count(own_side) <= 2 or _alive_count(opp_side) <= 2:
            score += (my_hp - opp_hp) * BoardValueV1.ENDGAME_WEIGHT

        if action is not None:
            kind = getattr(action, "action_type", None)
            if kind == ACTION_MOVE and best_move is not None:
                score += 140.0
            elif kind == ACTION_SWITCH:
                score -= 60.0

        return float(score)

    @staticmethod
    def breakdown(state, me, opp, side_name="player", action=None) -> BoardScoreBreakdown:
        my_hp = _hp_ratio(me)
        opp_hp = _hp_ratio(opp)
        best_move, best_raw, best_ko = _best_move_stats(state, me, opp)
        opp_threat = _opp_threat_into(state, me, opp)
        tempo = _field_tempo_bonus(state, me, opp, side_name)

        own_side = state.player_side if side_name == "player" else state.opponent_side
        opp_side = state.opponent_side if side_name == "player" else state.player_side
        hazards = -_entry_hazard_penalty(own_side) + 0.5 * _entry_hazard_penalty(opp_side)
        alive_diff = _alive_count(own_side) - _alive_count(opp_side)

        endgame = 0.0
        if _alive_count(own_side) <= 2 or _alive_count(opp_side) <= 2:
            endgame = (my_hp - opp_hp) * BoardValueV1.ENDGAME_WEIGHT

        total = (
            my_hp * BoardValueV1.OWN_HP_WEIGHT
            - opp_hp * BoardValueV1.OPP_HP_WEIGHT
            + best_raw * BoardValueV1.DAMAGE_WEIGHT
            + best_ko * BoardValueV1.KO_WEIGHT
            - opp_threat * BoardValueV1.THREAT_WEIGHT
            + tempo * BoardValueV1.TEMPO_WEIGHT
            + hazards * BoardValueV1.HAZARD_WEIGHT
            + alive_diff * BoardValueV1.ALIVE_WEIGHT
            + endgame
        )

        return BoardScoreBreakdown(
            own_hp=my_hp,
            opp_hp=opp_hp,
            damage=best_raw,
            ko=best_ko,
            threat=opp_threat,
            tempo=tempo,
            hazards=hazards,
            alive=alive_diff,
            endgame=endgame,
            total=float(total),
        )


def board_value(state, me, opp, side_name="player", action=None) -> float:
    return BoardValueV1.evaluate(state, me, opp, side_name=side_name, action=action)


def board_breakdown(state, me, opp, side_name="player", action=None) -> BoardScoreBreakdown:
    return BoardValueV1.breakdown(state, me, opp, side_name=side_name, action=action)
''')

file_path.write_text(content, encoding="utf-8")
print("created:", file_path)

created: /content/engine/board_value_v1.py


In [ ]:
import engine.board_value_v1 as bv
print(hasattr(bv, "board_value"))
print(hasattr(bv, "BoardValueV1"))

True
True


In [ ]:
import engine.decision_manager_v1 as dm
from engine.board_value_v1 import board_value

def _board_bonus(state, action, side_name="player"):
    try:
        me = state.player if side_name == "player" else state.opponent
        opp = state.opponent if side_name == "player" else state.player
        return float(board_value(state, me, opp, side_name=side_name, action=action))
    except Exception:
        return 0.0


def rank_actions_v61(state, actions, side_name="player"):
    base_ranked = dm.rank_actions_v60(state, actions, side_name)
    rescored = []

    for base_score, action in base_ranked:
        bonus = 0.03 * _board_bonus(state, action, side_name)
        rescored.append((float(base_score) + bonus, action))

    rescored.sort(key=lambda x: x[0], reverse=True)
    return rescored


def choose_best_v61(state, actions, side_name="player"):
    ranked = rank_actions_v61(state, actions, side_name)
    if not ranked:
        state._dm_last_action_kind = None
        return None, -100000.0

    best_action, best_score = ranked[0][1], float(ranked[0][0])
    state._dm_last_action_kind = getattr(best_action, "action_type", None)
    return best_action, best_score


dm.rank_actions_v61 = rank_actions_v61
dm.choose_best_v61 = choose_best_v61

print("v61 board-value bridge ready")

v61 board-value bridge ready


In [ ]:
print(dm.rank_actions_v61(state, actions, "player")[:3])

NameError: name 'actions' is not defined

In [ ]:
from battle.action import ACTION_MOVE, ACTION_SWITCH
import engine.decision_manager_v1 as dm

class MockAction:
    def __init__(self, action_type, move=None, switch_idx=None, priority=0):
        self.action_type = action_type
        self.move = move
        self.switch_index = switch_idx
        self.priority = priority

actions = []

# 技
for mv in getattr(state.player, "moves", []) or []:
    if mv:
        actions.append(MockAction(ACTION_MOVE, move=mv))

# 交代
for i, p in enumerate(getattr(state.player_side, "team", []) or []):
    if i != getattr(state.player_side, "active_index", 0) and not getattr(p, "fainted", False):
        actions.append(MockAction(ACTION_SWITCH, switch_idx=i))

print("actions =", len(actions))
print(dm.rank_actions_v61(state, actions, "player")[:3])

actions = 4
[(5000.0, <__main__.MockAction object at 0x7e88e4d66ba0>), (5000.0, <__main__.MockAction object at 0x7e88e4d67800>)]


In [ ]:
for a in actions:
    try:
        print(
            getattr(a, "action_type", None),
            getattr(getattr(a, "move", None), "name", None),
            getattr(a, "switch_index", None),
            _board_bonus(state, a, "player")
        )
    except Exception as e:
        print("ERROR:", e)

move None None 0.0
move None None 0.0
switch None 1 0.0
switch None 2 0.0


In [ ]:
import engine.board_value_v1 as bv

print(dir(bv))

['ACTION_MOVE', 'ACTION_SWITCH', 'BoardScoreBreakdown', 'BoardValueV1', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_alive_count', '_best_move_stats', '_entry_hazard_penalty', '_field_tempo_bonus', '_hp_ratio', '_opp_threat_into', '_profile_damage', 'annotations', 'board_breakdown', 'board_value', 'calculate_speed', 'damage_rolls', 'dataclass']


In [ ]:
import traceback
from engine.board_value_v1 import board_value

for a in actions:
    print("=" * 60)
    try:
        result = board_value(
            state,
            state.player,
            state.opponent,
            side_name="player",
            action=a,
        )
        print("RESULT =", result)
    except Exception:
        traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_1903/803195167.py", line 7, in <cell line: 0>
    result = board_value(
             ^^^^^^^^^^^^
  File "/content/engine/board_value_v1.py", line 217, in board_value
    return BoardValueV1.evaluate(state, me, opp, side_name=side_name, action=action)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/engine/board_value_v1.py", line 143, in evaluate
    tempo = _field_tempo_bonus(state, me, opp, side_name)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/engine/board_value_v1.py", line 99, in _field_tempo_bonus
    me_speed = calculate_speed(me, state.field, side_name)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/engine/stat_core_v2.py", line 54, in calculate_speed
    speed = get_modified_stat(pokemon, "spe")
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/engine/stat_core_v2.py", line 45, in get_modified_s

In [ ]:
class MockPokemon:
    def __init__(self, name):
        self.name = name

        self.base_stats = {
            "hp": 100,
            "atk": 100,
            "def": 100,
            "spa": 100,
            "spd": 100,
            "spe": 100,
        }

        self.stat_stages = {
            "atk": 0,
            "def": 0,
            "spa": 0,
            "spd": 0,
            "spe": 0,
        }

        self.types = []
        self.hp = 100
        self.max_hp = 100

In [ ]:
class MockPokemon:
    def __init__(self, name):
        self.name = name

        self.base_stats = {
            "hp": 100,
            "atk": 100,
            "def": 100,
            "spa": 100,
            "spd": 100,
            "spe": 100,
        }

        # 個体値
        self.ivs = {
            "hp": 31,
            "atk": 31,
            "def": 31,
            "spa": 31,
            "spd": 31,
            "spe": 31,
        }

        # 努力値
        self.evs = {
            "hp": 0,
            "atk": 0,
            "def": 0,
            "spa": 0,
            "spd": 0,
            "spe": 0,
        }

        self.nature = "serious"

        self.stat_stages = {
            "atk": 0,
            "def": 0,
            "spa": 0,
            "spd": 0,
            "spe": 0,
        }

        self.types = ["normal"]
        self.hp = 100
        self.max_hp = 100
        self.status = None
        self.ability = None
        self.item = None
        self.moves = []

In [ ]:
# AIフォルダ作成
import os

os.makedirs("/content/ai", exist_ok=True)

# move_selector.py 作成
move_selector_code = r'''
from engine.board_value_v1 import board_value


class MoveSelector:
    """
    盤面評価型の簡易AI
    """

    def choose_move(self, state, pokemon, opponent, moves):

        best_move = None
        best_score = -999999

        for move in moves:

            score = self.simulate(
                state,
                pokemon,
                opponent,
                move
            )

            if score > best_score:
                best_score = score
                best_move = move

        return best_move


    def simulate(self, state, pokemon, opponent, move):

        return board_value(
            state,
            pokemon,
            opponent,
            action=move
        )
'''

with open("/content/ai/move_selector.py", "w") as f:
    f.write(move_selector_code)


# battle_ai.py 作成
battle_ai_code = r'''
from ai.move_selector import MoveSelector


class BattleAI:

    def __init__(self):
        self.selector = MoveSelector()


    def decide(self, state, me, opponent):

        move = self.selector.choose_move(
            state,
            me,
            opponent,
            me.moves
        )

        return {
            "type": "move",
            "move": move
        }
'''

with open("/content/ai/battle_ai.py", "w") as f:
    f.write(battle_ai_code)


print("AIファイル作成完了")
print("/content/ai/move_selector.py")
print("/content/ai/battle_ai.py")

AIファイル作成完了
/content/ai/move_selector.py
/content/ai/battle_ai.py


In [ ]:
# =====================================
# BattleAI 動作テスト
# =====================================

from ai.battle_ai import BattleAI


# 仮の技データ
class MockMove:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name


# 技を持たせる
me.moves = [
    MockMove("かえんほうしゃ"),
    MockMove("10まんボルト"),
    MockMove("れいとうビーム"),
    MockMove("まもる")
]


# AI作成
ai = BattleAI()


# 判断
try:
    result = ai.decide(
        state,
        me,
        opp
    )

    print("===================")
    print("AI RESULT")
    print("===================")
    print(result)

except Exception as e:
    print("ERROR")
    print(type(e).__name__)
    print(e)

ERROR
AttributeError
'MockPokemon' object has no attribute 'ivs'


In [ ]:
# =====================================
# MockPokemon 更新
# =====================================

class MockPokemon:
    def __init__(self, name):
        self.name = name

        self.base_stats = {
            "hp": 100,
            "atk": 100,
            "def": 100,
            "spa": 100,
            "spd": 100,
            "spe": 100,
        }

        self.ivs = {
            "hp": 31,
            "atk": 31,
            "def": 31,
            "spa": 31,
            "spd": 31,
            "spe": 31,
        }

        self.evs = {
            "hp": 0,
            "atk": 0,
            "def": 0,
            "spa": 0,
            "spd": 0,
            "spe": 0,
        }

        self.nature = "serious"

        self.stat_stages = {
            "atk": 0,
            "def": 0,
            "spa": 0,
            "spd": 0,
            "spe": 0,
        }

        self.types = ["normal"]
        self.hp = 100
        self.max_hp = 100
        self.status = None
        self.ability = None
        self.item = None
        self.moves = []


# 作り直し
me = MockPokemon("Pikachu")
opp = MockPokemon("Charizard")

print("MockPokemon更新完了")

MockPokemon更新完了


In [ ]:
# =====================================
# BattleAI 再テスト
# =====================================

from ai.battle_ai import BattleAI


class MockMove:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name


# 技を設定
me.moves = [
    MockMove("かえんほうしゃ"),
    MockMove("10まんボルト"),
    MockMove("れいとうビーム"),
    MockMove("まもる")
]


# AI起動
ai = BattleAI()


try:
    result = ai.decide(
        state,
        me,
        opp
    )

    print("===================")
    print("AI RESULT")
    print("===================")
    print(result)

except Exception as e:
    print("===================")
    print("ERROR")
    print("===================")
    print(type(e).__name__)
    print(e)

ERROR
AttributeError
'MockPokemon' object has no attribute 'level'


In [ ]:
# =====================================
# MockPokemon 完全更新
# =====================================

class MockPokemon:
    def __init__(self, name):
        self.name = name
        self.level = 50

        # 種族値
        self.base_stats = {
            "hp": 100,
            "atk": 100,
            "def": 100,
            "spa": 100,
            "spd": 100,
            "spe": 100,
        }

        # 個体値
        self.ivs = {
            "hp": 31,
            "atk": 31,
            "def": 31,
            "spa": 31,
            "spd": 31,
            "spe": 31,
        }

        # 努力値
        self.evs = {
            "hp": 0,
            "atk": 0,
            "def": 0,
            "spa": 0,
            "spd": 0,
            "spe": 0,
        }

        # 性格
        self.nature = "serious"

        # ランク補正
        self.stat_stages = {
            "atk": 0,
            "def": 0,
            "spa": 0,
            "spd": 0,
            "spe": 0,
        }

        # 戦闘情報
        self.types = ["normal"]
        self.hp = 100
        self.max_hp = 100
        self.status = None

        self.ability = None
        self.item = None

        self.moves = []


# 作り直し
me = MockPokemon("Pikachu")
opp = MockPokemon("Charizard")


print("MockPokemon v2 loaded")
print(me.name, me.level)
print(opp.name, opp.level)

MockPokemon v2 loaded
Pikachu 50
Charizard 50


In [ ]:
# =====================================
# BattleAI 動作確認 v2
# =====================================

from ai.battle_ai import BattleAI


class MockMove:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name


# 技セット
me.moves = [
    MockMove("かえんほうしゃ"),
    MockMove("10まんボルト"),
    MockMove("れいとうビーム"),
    MockMove("まもる")
]


# AI生成
ai = BattleAI()


try:
    result = ai.decide(
        state,
        me,
        opp
    )

    print("===================")
    print("AI RESULT")
    print("===================")
    print(result)

except Exception as e:
    print("===================")
    print("ERROR")
    print("===================")
    print(type(e).__name__)
    print(e)

ERROR
AttributeError
'MockState' object has no attribute 'player_side'


In [ ]:
# =====================================
# MockState 更新 v2
# =====================================

class MockField:
    def __init__(self):
        self.weather = None
        self.terrain = None


class MockSide:
    def __init__(self, name):
        self.name = name


class MockState:
    def __init__(self):
        self.field = MockField()

        # player / opponent 情報
        self.player_side = MockSide("player")
        self.opponent_side = MockSide("opponent")


# 作り直し
state = MockState()


print("MockState v2 loaded")
print(state.player_side.name)
print(state.opponent_side.name)

MockState v2 loaded
player
opponent


In [ ]:
# =====================================
# BattleAI 動作確認 v3
# =====================================

from ai.battle_ai import BattleAI


class MockMove:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name


# 技設定
me.moves = [
    MockMove("かえんほうしゃ"),
    MockMove("10まんボルト"),
    MockMove("れいとうビーム"),
    MockMove("まもる")
]


# AI起動
ai = BattleAI()


try:
    result = ai.decide(
        state,
        me,
        opp
    )

    print("===================")
    print("AI RESULT")
    print("===================")
    print(result)

except Exception as e:
    print("===================")
    print("ERROR")
    print("===================")
    print(type(e).__name__)
    print(e)

AI RESULT
{'type': 'move', 'move': かえんほうしゃ}


In [ ]:
# プロジェクト診断
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".py"):
            print(os.path.join(root,file))

/content/utils/exporter.py
/content/utils/formatters.py
/content/utils/battle_analyzer.py
/content/utils/__init__.py
/content/utils/benchmark.py
/content/utils/testing.py
/content/tests/__init__.py
/content/data/regulation_integration_v1.py
/content/data/pokemon_database.py
/content/data/__init__.py
/content/data/type_chart.py
/content/data/move_database.py
/content/data/regulation_manager_v1.py
/content/learning/replay_buffer.py
/content/learning/ai_trainer.py
/content/learning/__init__.py
/content/learning/puct.py
/content/learning/policy.py
/content/learning/self_play.py
/content/learning/metrics.py
/content/learning/value.py
/content/engine/exact_mechanics.py
/content/engine/stat_engine.py
/content/engine/opponent_model.py
/content/engine/evaluation_core_v3.py
/content/engine/switch_value_v1.py
/content/engine/advanced_evaluation.py
/content/engine/action_value_v1.py
/content/engine/effect_engine.py
/content/engine/__init__.py
/content/engine/stat_core_v2.py
/content/engine/board_v

In [ ]:
# ==========================================
# PokeChamp AI Project Diagnostic v1
# ==========================================

import os
import importlib


BASE = "/content"

print("=" * 50)
print(" PokeChamp AI Diagnostic")
print("=" * 50)


# -----------------------------
# ファイル存在チェック
# -----------------------------

targets = [
    "battle.battle_state",
    "battle.pokemon",
    "battle.action",
    "engine.damage_engine",
    "engine.board_value_v1",
    "engine.search_engine",
    "engine.decision_manager_v1",
    "ai.mcts",
    "ai.battle_ai",
    "learning.self_play",
]


print("\n[MODULE CHECK]")

missing = []

for module in targets:
    try:
        importlib.import_module(module)
        print("OK  ", module)

    except Exception as e:
        print("FAIL", module)
        print("   ", type(e).__name__, e)
        missing.append(module)


# -----------------------------
# ファイル数確認
# -----------------------------

print("\n[FILE CHECK]")

count = 0

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith(".py"):
            count += 1

print("Python files:", count)


# -----------------------------
# AI主要ファイル確認
# -----------------------------

print("\n[AI COMPONENT CHECK]")

components = {
    "MCTS": "/content/ai/mcts.py",
    "BattleAI": "/content/ai/battle_ai.py",
    "Damage": "/content/engine/damage_engine.py",
    "Evaluation": "/content/engine/board_value_v1.py",
    "Pokemon": "/content/battle/pokemon.py",
    "State": "/content/battle/battle_state.py",
}


for name, path in components.items():

    if os.path.exists(path):
        print("OK  ", name)
    else:
        print("MISS", name)


# -----------------------------
# 結果
# -----------------------------

print("\n" + "=" * 50)
print(" Diagnostic Finished")
print("=" * 50)

if missing:
    print("\n修正候補:")
    for m in missing:
        print("-", m)
else:
    print("\n主要モジュールは読み込み成功")

 PokeChamp AI Diagnostic

[MODULE CHECK]
OK   battle.battle_state
OK   battle.pokemon
OK   battle.action
OK   engine.damage_engine
OK   engine.board_value_v1
OK   engine.search_engine
OK   engine.decision_manager_v1
OK   ai.mcts
OK   ai.battle_ai
OK   learning.self_play

[FILE CHECK]
Python files: 73

[AI COMPONENT CHECK]
OK   MCTS
OK   BattleAI
OK   Damage
OK   Evaluation
OK   Pokemon
OK   State

 Diagnostic Finished

主要モジュールは読み込み成功


In [ ]:
# ==========================================
# PokeChamp AI Integration Test v1
# ==========================================

print("=" * 50)
print(" PokeChamp AI Integration Test")
print("=" * 50)


errors = []


# BattleAI
try:
    from ai.battle_ai import BattleAI
    ai = BattleAI()
    print("OK  BattleAI instance")

except Exception as e:
    print("FAIL BattleAI")
    print(type(e).__name__, e)
    errors.append(("BattleAI", e))


# MCTS
try:
    from ai.mcts import MCTS
    print("OK  MCTS import")

except Exception as e:
    print("FAIL MCTS")
    print(type(e).__name__, e)
    errors.append(("MCTS", e))


# State
try:
    from battle.battle_state import BattleState
    print("OK  BattleState import")

except Exception as e:
    print("FAIL BattleState")
    print(type(e).__name__, e)
    errors.append(("BattleState", e))


# Pokemon
try:
    from battle.pokemon import Pokemon
    print("OK  Pokemon import")

except Exception as e:
    print("FAIL Pokemon")
    print(type(e).__name__, e)
    errors.append(("Pokemon", e))


print("\n" + "=" * 50)

if errors:
    print("修正必要:")
    for name, err in errors:
        print("-", name, err)

else:
    print("基本接続OK")

 PokeChamp AI Integration Test
OK  BattleAI instance
FAIL MCTS
ImportError cannot import name 'MCTS' from 'ai.mcts' (/content/ai/mcts.py)
OK  BattleState import
OK  Pokemon import

修正必要:
- MCTS cannot import name 'MCTS' from 'ai.mcts' (/content/ai/mcts.py)


In [ ]:
# ==========================================
# mcts.py 解析
# ==========================================

path = "/content/ai/mcts.py"

print("=" * 50)
print(path)
print("=" * 50)

with open(path, "r") as f:
    code = f.read()

print(code[:5000])

/content/ai/mcts.py
from __future__ import annotations
from dataclasses import dataclass, field
from copy import deepcopy
import math
import random


@dataclass
class Node:
    state: object
    parent: "Node | None" = None
    action: object | None = None
    children: list["Node"] = field(default_factory=list)
    visits: int = 0
    value: float = 0.0
    untried_actions: list = field(default_factory=list)


def uct_score(parent_visits, child_value, child_visits, c=1.414):
    if child_visits == 0:
        return float("inf")
    return (child_value / child_visits) + c * math.sqrt(math.log(parent_visits + 1) / child_visits)


def mcts_choose_action(state, actions, rollout_fn, iterations=100):
    root = Node(state=deepcopy(state), untried_actions=list(actions))
    if not actions:
        return None

    for _ in range(iterations):
        node = root
        while node.children and not node.untried_actions:
            node = max(node.children, key=lambda ch: uct_score(node.visits

In [ ]:
# ==========================================
# MCTS Class Wrapper追加
# ==========================================

path = "/content/ai/mcts.py"

with open(path, "a") as f:
    f.write(r'''


class MCTS:
    """
    Class interface wrapper for existing MCTS function
    """

    def __init__(self, iterations=100):
        self.iterations = iterations


    def search(self, state, actions, rollout_fn):
        return mcts_choose_action(
            state,
            actions,
            rollout_fn,
            iterations=self.iterations
        )


    def choose_action(self, state, actions, rollout_fn):
        return self.search(
            state,
            actions,
            rollout_fn
        )
''')

print("MCTS wrapper added")

MCTS wrapper added


In [ ]:
# ==========================================
# MCTS Wrapper Test
# ==========================================

from ai.mcts import MCTS

try:
    mcts = MCTS(iterations=10)

    print("===================")
    print("MCTS TEST")
    print("===================")
    print("Instance:", mcts)
    print("Iterations:", mcts.iterations)
    print("STATUS: OK")

except Exception as e:
    print("===================")
    print("ERROR")
    print("===================")
    print(type(e).__name__)
    print(e)

ImportError: cannot import name 'MCTS' from 'ai.mcts' (/content/ai/mcts.py)

In [ ]:
# ==========================================
# mcts.py 強制リロード
# ==========================================

import importlib
import ai.mcts

importlib.reload(ai.mcts)

from ai.mcts import MCTS

print("MCTS reload OK")
print(MCTS)

MCTS reload OK
<class 'ai.mcts.MCTS'>


In [ ]:
# ==========================================
# PokeChamp AI Full Connection Test v1
# ==========================================

print("=" * 60)
print(" PokeChamp AI Full Connection Test")
print("=" * 60)


errors = []


# imports
try:
    from ai.battle_ai import BattleAI
    from ai.mcts import MCTS
    from battle.battle_state import BattleState
    from battle.pokemon import Pokemon
    from battle.action import Action

    print("OK imports")

except Exception as e:
    print("IMPORT ERROR")
    print(type(e).__name__, e)
    errors.append(e)


# instance test
try:
    ai = BattleAI()
    mcts = MCTS(iterations=10)

    print("OK AI instance")
    print("OK MCTS instance")

except Exception as e:
    print("INSTANCE ERROR")
    print(type(e).__name__, e)
    errors.append(e)


print("=" * 60)

if errors:
    print("STATUS: NEED FIX")
else:
    print("STATUS: CORE READY")

 PokeChamp AI Full Connection Test
OK imports
OK AI instance
OK MCTS instance
STATUS: CORE READY


In [ ]:
# ==========================================
# PokeChamp AI Battle Decision Test v1
# ==========================================

print("=" * 60)
print(" PokeChamp AI Battle Decision Test")
print("=" * 60)

errors = []


# -----------------------------
# Import
# -----------------------------

try:
    from ai.battle_ai import BattleAI
    from battle.action import Action

    print("OK imports")

except Exception as e:
    print("IMPORT ERROR")
    print(type(e).__name__, e)
    errors.append(e)


# -----------------------------
# AI生成
# -----------------------------

try:
    ai = BattleAI()

    print("OK BattleAI created")

except Exception as e:
    print("AI CREATE ERROR")
    print(type(e).__name__, e)
    errors.append(e)


# -----------------------------
# 仮想State確認
# -----------------------------

try:
    print("State type:", type(state).__name__)
    print("My Pokemon:", me.name)
    print("Opponent:", opp.name)

except Exception as e:
    print("STATE ERROR")
    print(type(e).__name__, e)
    errors.append(e)


# -----------------------------
# AI判断
# -----------------------------

try:

    result = ai.decide(
        state,
        me,
        opp
    )

    print("\n===================")
    print("AI DECISION RESULT")
    print("===================")
    print(result)

except Exception as e:

    print("\n===================")
    print("DECISION ERROR")
    print("===================")
    print(type(e).__name__)
    print(e)

    errors.append(e)



print("\n" + "=" * 60)

if errors:
    print("STATUS: NEED FIX")
else:
    print("STATUS: AI CAN DECIDE")

 PokeChamp AI Battle Decision Test
OK imports
OK BattleAI created
State type: MockState
My Pokemon: Pikachu
Opponent: Charizard

AI DECISION RESULT
{'type': 'move', 'move': かえんほうしゃ}

STATUS: AI CAN DECIDE


In [ ]:
# ==========================================
# PokeChamp AI MCTS Usage Test v1
# ==========================================

print("=" * 60)
print(" PokeChamp AI MCTS Usage Test")
print("=" * 60)


# MCTS import
try:
    from ai.mcts import MCTS, mcts_choose_action

    print("OK MCTS import")

except Exception as e:
    print("MCTS IMPORT ERROR")
    print(type(e).__name__, e)


# MCTS単体テスト

try:

    test_actions = [
        "move_A",
        "move_B",
        "move_C"
    ]


    class TestState:
        def __init__(self):
            self.value = 0


    def rollout(state, action):

        scores = {
            "move_A": 1,
            "move_B": 5,
            "move_C": 3
        }

        return scores[action]


    result = mcts_choose_action(
        TestState(),
        test_actions,
        rollout,
        iterations=50
    )


    print("\n===================")
    print("MCTS RESULT")
    print("===================")
    print("Chosen:", result)


except Exception as e:

    print("\nMCTS TEST ERROR")
    print(type(e).__name__)
    print(e)



print("\n" + "=" * 60)
print("TEST FINISHED")
print("=" * 60)

 PokeChamp AI MCTS Usage Test
OK MCTS import

MCTS TEST ERROR
KeyError
None

TEST FINISHED


In [ ]:
# ==========================================
# MCTS Interface Test v2
# ==========================================

from ai.mcts import mcts_choose_action


actions = [
    "attack",
    "defend",
    "switch"
]


class TestState:
    pass



def rollout_fn(state, action):

    # 終了評価の場合
    if action is None:
        return 0.0

    scores = {
        "attack": 10.0,
        "defend": 3.0,
        "switch": 1.0
    }

    return scores[action]



result = mcts_choose_action(
    TestState(),
    actions,
    rollout_fn,
    iterations=100
)


print("===================")
print("MCTS RESULT")
print("===================")
print(result)

MCTS RESULT
attack


In [ ]:
# ==========================================
# PokeChamp BattleAI -> MCTS Connection Test
# ==========================================

import inspect

print("=" * 60)
print(" BattleAI -> MCTS Connection Diagnostic")
print("=" * 60)


# BattleAI読み込み
from ai.battle_ai import BattleAI


# ソース確認
try:

    source = inspect.getsource(BattleAI)

    print("\n[BattleAI SOURCE CHECK]")

    if "MCTS" in source:
        print("FOUND: MCTS reference")

    else:
        print("NOT FOUND: MCTS reference")


    if "search" in source:
        print("FOUND: search() call")

    else:
        print("NOT FOUND: search() call")


    if "choose_action" in source:
        print("FOUND: choose_action() call")

    else:
        print("NOT FOUND: choose_action() call")


except Exception as e:

    print("SOURCE ERROR")
    print(type(e).__name__, e)



# 実際のAIインスタンス確認

print("\n[INSTANCE CHECK]")

try:

    ai = BattleAI()

    print("AI:",
          type(ai).__name__)


    attrs = dir(ai)

    print("\nAvailable methods:")

    for a in attrs:
        if not a.startswith("_"):
            print("-", a)


except Exception as e:

    print("INSTANCE ERROR")
    print(type(e).__name__, e)



print("\n" + "=" * 60)
print("Diagnostic Finished")
print("=" * 60)

 BattleAI -> MCTS Connection Diagnostic

[BattleAI SOURCE CHECK]
NOT FOUND: MCTS reference
NOT FOUND: search() call
NOT FOUND: choose_action() call

[INSTANCE CHECK]
AI: BattleAI

Available methods:
- decide
- selector

Diagnostic Finished


In [ ]:
# ==========================================
# battle_ai.py source export
# ==========================================

path = "/content/ai/battle_ai.py"

print("=" * 60)
print(path)
print("=" * 60)

with open(path, "r") as f:
    code = f.read()

print(code)

/content/ai/battle_ai.py

from ai.move_selector import MoveSelector


class BattleAI:

    def __init__(self):
        self.selector = MoveSelector()


    def decide(self, state, me, opponent):

        move = self.selector.choose_move(
            state,
            me,
            opponent,
            me.moves
        )

        return {
            "type": "move",
            "move": move
        }



In [ ]:
# ==========================================
# BattleAI MCTS Integration v1
# ==========================================

path = "/content/ai/battle_ai.py"


code = r'''
from ai.move_selector import MoveSelector
from ai.mcts import MCTS


class BattleAI:

    def __init__(self, iterations=100):

        self.selector = MoveSelector()

        # MCTS追加
        self.mcts = MCTS(
            iterations=iterations
        )


    def decide(self, state, me, opponent):

        try:

            # MCTS用候補行動
            actions = list(me.moves)


            if actions:

                def rollout(sim_state, action):

                    # 現段階では評価関数を利用
                    return 0.0


                best = self.mcts.choose_action(
                    state,
                    actions,
                    rollout
                )


                if best is not None:

                    return {
                        "type": "move",
                        "move": best
                    }


        except Exception:

            pass


        # fallback
        move = self.selector.choose_move(
            state,
            me,
            opponent,
            me.moves
        )


        return {
            "type": "move",
            "move": move
        }
'''

with open(path, "w") as f:
    f.write(code)


print("BattleAI MCTS version written")

BattleAI MCTS version written


In [ ]:
import importlib
import ai.battle_ai

importlib.reload(ai.battle_ai)

from ai.battle_ai import BattleAI

print("BattleAI reload OK")

BattleAI reload OK


In [ ]:
# ==========================================
# BattleAI MCTS Decision Test v1
# ==========================================

import importlib
import ai.battle_ai

importlib.reload(ai.battle_ai)

from ai.battle_ai import BattleAI


print("=" * 60)
print(" BattleAI MCTS Decision Test")
print("=" * 60)


try:

    ai = BattleAI(iterations=50)


    result = ai.decide(
        state,
        me,
        opp
    )


    print("\n===================")
    print("RESULT")
    print("===================")
    print(result)


    if result.get("move") is not None:
        print("\nSTATUS: AI DECISION OK")
    else:
        print("\nSTATUS: NO MOVE")


except Exception as e:

    print("\nERROR")
    print(type(e).__name__)
    print(e)

 BattleAI MCTS Decision Test

RESULT
{'type': 'move', 'move': まもる}

STATUS: AI DECISION OK


In [ ]:
# ==========================================
# MCTS Rollout Engine v1
# ==========================================

import os

os.makedirs("/content/engine", exist_ok=True)


code = r'''
from engine.board_value_v1 import board_value


class MCTSRollout:

    """
    MCTS用の簡易シミュレーション評価

    action:
        技や行動

    return:
        数値評価
    """


    def evaluate(
        self,
        state,
        me,
        opponent,
        action
    ):

        try:

            # 仮想評価
            score = board_value(
                state,
                me,
                opponent,
                action=action
            )

            return float(score)


        except Exception:

            # 失敗時は最低評価
            return -9999.0



    def rollout(
        self,
        state,
        me,
        opponent,
        action
    ):

        return self.evaluate(
            state,
            me,
            opponent,
            action
        )
'''


with open(
    "/content/engine/mcts_rollout.py",
    "w"
) as f:
    f.write(code)


print("MCTS Rollout Engine created")

MCTS Rollout Engine created


In [ ]:
# ==========================================
# MCTS Rollout Import Test
# ==========================================

from engine.mcts_rollout import MCTSRollout

try:

    rollout = MCTSRollout()

    print("MCTSRollout OK")
    print(type(rollout).__name__)

except Exception as e:

    print("ERROR")
    print(type(e).__name__)
    print(e)

MCTSRollout OK
MCTSRollout


In [ ]:
# ==========================================
# BattleAI MCTS Rollout Connection v2
# ==========================================

path = "/content/ai/battle_ai.py"


code = r'''
from ai.move_selector import MoveSelector
from ai.mcts import MCTS
from engine.mcts_rollout import MCTSRollout


class BattleAI:

    def __init__(self, iterations=100):

        self.selector = MoveSelector()

        self.mcts = MCTS(
            iterations=iterations
        )

        self.rollout_engine = MCTSRollout()



    def decide(self, state, me, opponent):

        try:

            actions = list(me.moves)


            if actions:


                def rollout(sim_state, action):

                    return self.rollout_engine.rollout(
                        sim_state,
                        me,
                        opponent,
                        action
                    )


                best = self.mcts.choose_action(
                    state,
                    actions,
                    rollout
                )


                if best is not None:

                    return {
                        "type": "move",
                        "move": best
                    }



        except Exception as e:

            print(
                "MCTS fallback:",
                type(e).__name__,
                e
            )


        # fallback
        move = self.selector.choose_move(
            state,
            me,
            opponent,
            me.moves
        )


        return {
            "type": "move",
            "move": move
        }
'''


with open(path, "w") as f:
    f.write(code)


print("BattleAI MCTS Rollout Connected")

BattleAI MCTS Rollout Connected


In [ ]:
import importlib
import ai.battle_ai

importlib.reload(ai.battle_ai)

from ai.battle_ai import BattleAI

print("BattleAI reload OK")

BattleAI reload OK


In [ ]:
# ==========================================
# BattleAI MCTS Rollout Decision Test v2
# ==========================================

print("=" * 60)
print(" BattleAI MCTS Rollout Decision Test")
print("=" * 60)


from ai.battle_ai import BattleAI


try:

    ai = BattleAI(iterations=50)


    result = ai.decide(
        state,
        me,
        opp
    )


    print("\n===================")
    print("RESULT")
    print("===================")
    print(result)


    if result.get("move") is not None:
        print("\nSTATUS: MCTS + EVALUATION OK")
    else:
        print("\nSTATUS: NO MOVE")


except Exception as e:

    print("\n===================")
    print("ERROR")
    print("===================")
    print(type(e).__name__)
    print(e)

 BattleAI MCTS Rollout Decision Test

RESULT
{'type': 'move', 'move': まもる}

STATUS: MCTS + EVALUATION OK


In [ ]:
# ==========================================
# Move Evaluation Diagnostic
# ==========================================

from engine.mcts_rollout import MCTSRollout


rollout = MCTSRollout()


print("=" * 60)
print(" Move Score Diagnostic")
print("=" * 60)


for move in me.moves:

    try:

        score = rollout.evaluate(
            state,
            me,
            opp,
            move
        )

        print(
            move,
            "=>",
            score
        )


    except Exception as e:

        print(
            move,
            "ERROR",
            type(e).__name__,
            e
        )


print("=" * 60)

 Move Score Diagnostic
かえんほうしゃ => 180.0
10まんボルト => 180.0
れいとうビーム => 180.0
まもる => 180.0


In [ ]:
# ==========================================
# Board Value Action Sensitivity Test
# ==========================================

from engine.board_value_v1 import board_value


print("=" * 60)
print(" Board Value Action Test")
print("=" * 60)


for move in me.moves:

    try:

        value = board_value(
            state,
            me,
            opp,
            action=move
        )

        print(
            move,
            "=>",
            value
        )


    except Exception as e:

        print(
            move,
            "ERROR",
            type(e).__name__,
            e
        )


print("=" * 60)

 Board Value Action Test
かえんほうしゃ => 180.0
10まんボルト => 180.0
れいとうビーム => 180.0
まもる => 180.0


In [ ]:
path="/content/engine/action_value_v1.py"

print("="*60)
print(path)
print("="*60)

with open(path,"r") as f:
    print(f.read())

/content/engine/action_value_v1.py
from __future__ import annotations

from engine.exact_mechanics import damage_rolls
from engine.stat_engine import calculate_speed


def _first_move_name(pokemon):
    for m in getattr(pokemon, "moves", []) or []:
        if m:
            return m
    return None


def action_value(state, action, side_name="player"):
    """
    Immediate action heuristic.

    Positive = better move
    Negative = bad move
    """

    if action is None:
        return -100000.0

    if side_name == "player":
        me = state.player
        opp = state.opponent
    else:
        me = state.opponent
        opp = state.player

    score = 0.0

    if getattr(action, "action_type", "") == "switch":

        hp_ratio = me.current_hp / max(1, me.max_hp)

        if hp_ratio < 0.30:
            score += 120

        else:
            score += 20

        return float(score)

    move = str(getattr(action, "move", "")).lower()

    try:
        profile = damage_rolls(
 

In [ ]:
# ==========================================
# MCTSRollout Action Value Integration
# ==========================================

path = "/content/engine/mcts_rollout.py"


code = r'''
from engine.board_value_v1 import board_value
from engine.action_value_v1 import action_value


class MCTSRollout:


    def evaluate(
        self,
        state,
        me,
        opponent,
        action
    ):

        score = 0.0


        # 盤面評価
        try:
            score += board_value(
                state,
                me,
                opponent,
                action=action
            )

        except Exception:
            pass



        # 技・行動評価
        try:

            score += action_value(
                state,
                action
            )

        except Exception:

            pass


        return float(score)



    def rollout(
        self,
        state,
        me,
        opponent,
        action
    ):

        return self.evaluate(
            state,
            me,
            opponent,
            action
        )
'''


with open(path,"w") as f:
    f.write(code)


print("MCTSRollout action_value connected")

MCTSRollout action_value connected


In [ ]:
# ==========================================
# MCTSRollout Reload Test
# ==========================================

import importlib
import engine.mcts_rollout

importlib.reload(engine.mcts_rollout)

from engine.mcts_rollout import MCTSRollout

print("MCTSRollout reload OK")

MCTSRollout reload OK


In [ ]:
# ==========================================
# Move Score Diagnostic v2
# ==========================================

from engine.mcts_rollout import MCTSRollout


rollout = MCTSRollout()

print("=" * 60)
print(" Move Score Diagnostic v2")
print("=" * 60)


for move in me.moves:

    score = rollout.evaluate(
        state,
        me,
        opp,
        move
    )

    print(
        move,
        "=>",
        score
    )


print("=" * 60)

 Move Score Diagnostic v2
かえんほうしゃ => 180.0
10まんボルト => 180.0
れいとうビーム => 180.0
まもる => 180.0


In [ ]:
# ==========================================
# action.py確認
# ==========================================

path="/content/battle/action.py"

print("="*60)
print(path)
print("="*60)

with open(path,"r") as f:
    print(f.read())

/content/battle/action.py
from __future__ import annotations
from dataclasses import dataclass

ACTION_MOVE = "move"
ACTION_SWITCH = "switch"


@dataclass(frozen=True)
class Action:
    action_type: str
    move: str | None = None
    switch_index: int | None = None



In [ ]:
# ==========================================
# BattleAI Action Wrapper Fix
# ==========================================

path = "/content/ai/battle_ai.py"

with open(path, "r") as f:
    code = f.read()

code = code.replace(
    "actions = list(me.moves)",
    """
from battle.action import Action, ACTION_MOVE

actions = [
    Action(
        action_type=ACTION_MOVE,
        move=getattr(m, "name", str(m))
    )
    for m in me.moves
]
"""
)

with open(path, "w") as f:
    f.write(code)

print("BattleAI Action wrapper added")

BattleAI Action wrapper added


In [ ]:
# ==========================================
# BattleAI Fix Indentation
# ==========================================

path = "/content/ai/battle_ai.py"


code = r'''
from ai.move_selector import MoveSelector
from ai.mcts import MCTS
from engine.mcts_rollout import MCTSRollout
from battle.action import Action, ACTION_MOVE


class BattleAI:

    def __init__(self, iterations=100):

        self.selector = MoveSelector()

        self.mcts = MCTS(
            iterations=iterations
        )

        self.rollout_engine = MCTSRollout()


    def decide(self, state, me, opponent):

        try:

            actions = [
                Action(
                    action_type=ACTION_MOVE,
                    move=getattr(m, "name", str(m))
                )
                for m in me.moves
            ]


            def rollout(sim_state, action):

                return self.rollout_engine.rollout(
                    sim_state,
                    me,
                    opponent,
                    action
                )


            best = self.mcts.choose_action(
                state,
                actions,
                rollout
            )


            if best is not None:

                return {
                    "type": "move",
                    "move": best.move
                }


        except Exception as e:

            print(
                "MCTS ERROR:",
                type(e).__name__,
                e
            )


        move = self.selector.choose_move(
            state,
            me,
            opponent,
            me.moves
        )


        return {
            "type": "move",
            "move": move
        }
'''


with open(path, "w") as f:
    f.write(code)


print("BattleAI fixed")

BattleAI fixed


In [ ]:
import importlib
import ai.battle_ai

importlib.reload(ai.battle_ai)

from ai.battle_ai import BattleAI

print("BattleAI reload OK")

BattleAI reload OK


In [ ]:
# ==========================================
# Move Score Diagnostic v3
# ==========================================

from engine.mcts_rollout import MCTSRollout
from battle.action import Action, ACTION_MOVE


rollout = MCTSRollout()


print("=" * 60)
print(" Move Score Diagnostic v3")
print("=" * 60)


for m in me.moves:

    action = Action(
        action_type=ACTION_MOVE,
        move=getattr(m, "name", str(m))
    )

    try:

        score = rollout.evaluate(
            state,
            me,
            opp,
            action
        )

        print(
            action.move,
            "=>",
            score
        )

    except Exception as e:

        print(
            action.move,
            "ERROR",
            type(e).__name__,
            e
        )


print("=" * 60)

 Move Score Diagnostic v3
かえんほうしゃ => 180.0
10まんボルト => 180.0
れいとうビーム => 180.0
まもる => 180.0


In [ ]:
# ==========================================
# damage_rolls check
# ==========================================

import inspect
from engine.exact_mechanics import damage_rolls


print("="*60)
print("damage_rolls signature")
print("="*60)

print(inspect.signature(damage_rolls))


print("\nSOURCE")

print(
    inspect.getsource(damage_rolls)[:2000]
)

damage_rolls signature
(attacker, defender, move_name: 'str', field=None) -> 'DamageProfile'

SOURCE
def damage_rolls(attacker, defender, move_name: str, field=None) -> DamageProfile:
    move = get_move(move_name)
    if move is None:
        # Unknown move fallback: do not crash the notebook.
        return DamageProfile(
            move=move_name,
            min_damage=0,
            max_damage=0,
            avg_damage=0.0,
            rolls=(0,) * 16,
            accuracy=0.0,
            ko_chance=0.0,
        )

    if move.category == "status":
        return DamageProfile(
            move=move.name,
            min_damage=0,
            max_damage=0,
            avg_damage=0.0,
            rolls=(0,) * 16,
            accuracy=1.0,
            ko_chance=0.0,
        )

    acc = accuracy_multiplier(attacker, defender, move)
    raw = base_damage(attacker, defender, move, field=field, crit=False)

    rolls = []
    hp = max(1, int(getattr(defender, "current_hp", 1) or 1))
 

In [ ]:
# ==========================================
# Direct action_value Test
# ==========================================

from engine.action_value_v1 import action_value
from battle.action import Action, ACTION_MOVE


print("="*60)
print("Direct action_value Test")
print("="*60)


for m in me.moves:

    action = Action(
        action_type=ACTION_MOVE,
        move=getattr(m, "name", str(m))
    )


    try:

        value = action_value(
            state,
            action
        )

        print(
            action.move,
            "=>",
            value
        )


    except Exception as e:

        print(
            action.move,
            "ERROR",
            type(e).__name__,
            e
        )


print("="*60)

Direct action_value Test
かえんほうしゃ ERROR AttributeError 'MockState' object has no attribute 'player'
10まんボルト ERROR AttributeError 'MockState' object has no attribute 'player'
れいとうビーム ERROR AttributeError 'MockState' object has no attribute 'player'
まもる ERROR AttributeError 'MockState' object has no attribute 'player'


In [ ]:
# ==========================================
# MockState Check
# ==========================================

print(type(state))

print("="*60)

for attr in dir(state):

    if not attr.startswith("_"):
        print(attr)

<class '__main__.MockState'>
field
opponent_side
player_side


In [ ]:
# ==========================================
# MockSide Check
# ==========================================

print("="*60)

print(type(state.player_side))

print("="*60)

for attr in dir(state.player_side):

    if not attr.startswith("_"):
        print(attr)

print("="*60)

print(type(state.opponent_side))

for attr in dir(state.opponent_side):

    if not attr.startswith("_"):
        print(attr)

<class '__main__.MockSide'>
name
<class '__main__.MockSide'>
name


In [ ]:
# ==========================================
# BattleState Structure Check
# ==========================================

from battle.battle_state import BattleState

print("="*60)
print("BattleState")
print("="*60)

print(BattleState)

print("\nAttributes:")

for a in dir(state):

    if not a.startswith("_"):
        print(a)

BattleState
<class 'battle.battle_state.BattleState'>

Attributes:
field
opponent
opponent_side
player
player_side


In [ ]:
# ==========================================
# Current State Type Check
# ==========================================

print(type(state))

print("player:",
      getattr(state, "player", None))

print("opponent:",
      getattr(state, "opponent", None))

<class '__main__.MockStateCompat'>
player: None
opponent: None


In [ ]:
# ==========================================
# Find BattleState Creation
# ==========================================

import inspect
from battle.battle_state import BattleState

print(inspect.signature(BattleState))

(player_side: 'object', opponent_side: 'object', turn: 'int' = 1, log: 'list[str]' = <factory>, field: 'FieldState' = <factory>) -> None


In [ ]:
# ==========================================
# Side Check
# ==========================================

from battle.side import Side
import inspect

print(inspect.signature(Side))

(team: 'list', active_index: 'int' = 0, side_conditions: 'dict[str, object]' = <factory>) -> None


In [ ]:
# ==========================================
# Real BattleState Action Value Test
# ==========================================

from battle.side import Side
from battle.battle_state import BattleState
from battle.field import FieldState
from engine.action_value_v1 import action_value
from battle.action import Action, ACTION_MOVE


# 既存のPokemonを使用
player_side_real = Side(
    team=[me]
)

opponent_side_real = Side(
    team=[opp]
)


real_state = BattleState(
    player_side=player_side_real,
    opponent_side=opponent_side_real,
    field=FieldState()
)


print("="*60)
print("BattleState Created")
print("="*60)

print(
    "player:",
    real_state.player.name
)

print(
    "opponent:",
    real_state.opponent.name
)


print("="*60)
print("Action Value Test")
print("="*60)


for move in real_state.player.moves:

    action = Action(
        action_type=ACTION_MOVE,
        move=getattr(move, "name", str(move))
    )

    score = action_value(
        real_state,
        action
    )

    print(
        action.move,
        "=>",
        score
    )


print("="*60)

BattleState Created
player: Pikachu
opponent: Charizard
Action Value Test


AttributeError: 'MockPokemon' object has no attribute 'current_hp'

In [ ]:
# ==========================================
# MockPokemon current_hp patch
# ==========================================

me.current_hp = me.max_hp
opp.current_hp = opp.max_hp

print("current_hp added")

print(
    me.name,
    me.current_hp,
    "/",
    me.max_hp
)

print(
    opp.name,
    opp.current_hp,
    "/",
    opp.max_hp
)

current_hp added
Pikachu 100 / 100
Charizard 100 / 100


In [ ]:
# ==========================================
# Action Value Test Retry
# ==========================================

from engine.action_value_v1 import action_value
from battle.action import Action, ACTION_MOVE


print("="*60)
print("Action Value Test Retry")
print("="*60)


for move in real_state.player.moves:

    action = Action(
        action_type=ACTION_MOVE,
        move=getattr(move, "name", str(move))
    )

    score = action_value(
        real_state,
        action
    )

    print(
        action.move,
        "=>",
        score
    )

Action Value Test Retry
かえんほうしゃ => 0.0
10まんボルト => 0.0
れいとうビーム => 0.0
まもる => 0.0


In [ ]:
# ==========================================
# Damage Roll Direct Test
# ==========================================

from engine.exact_mechanics import damage_rolls


print("="*60)
print("Direct Damage Test")
print("="*60)


for move in me.moves:

    name = getattr(move, "name", str(move))

    result = damage_rolls(
        me,
        opp,
        name,
        real_state.field
    )

    print(
        name,
        "damage:",
        result.avg_damage,
        "KO:",
        result.ko_chance
    )


print("="*60)

Direct Damage Test
かえんほうしゃ damage: 0.0 KO: 0.0
10まんボルト damage: 0.0 KO: 0.0
れいとうビーム damage: 0.0 KO: 0.0
まもる damage: 0.0 KO: 0.0


In [ ]:
# ==========================================
# base_damage Check
# ==========================================

import inspect
from engine.exact_mechanics import base_damage

print("="*60)
print(inspect.signature(base_damage))
print("="*60)

print(inspect.getsource(base_damage)[:3000])

(attacker, defender, move, field=None, crit: 'bool' = False) -> 'float'
def base_damage(attacker, defender, move, field=None, crit: bool = False) -> float:
    if move.category == "status":
        return 0.0

    atk_stat = get_modified_stat(attacker, "spa" if move.category == "special" else "atk")
    def_stat = get_modified_stat(defender, "spd" if move.category == "special" else "def")

    level_factor = (2 * attacker.level / 5) + 2
    base = (((level_factor * move.power * atk_stat / max(1, def_stat)) / 50) + 2)

    modifier = (
        stab(attacker, move.type)
        * type_multiplier(move.type, getattr(defender, "types", ()))
        * weather_modifier(move.type, field)
    )

    if getattr(attacker, "status", None) == "burn" and move.category == "physical" and getattr(attacker, "ability", "") != "guts":
        modifier *= CONFIG.burn_multiplier

    if crit:
        modifier *= crit_multiplier(attacker, defender, move)

    if getattr(defender, "volatile_status", {}).get("

In [ ]:
# ==========================================
# Stat Check
# ==========================================

from engine.stat_core_v2 import get_modified_stat


print("="*60)
print("STAT CHECK")
print("="*60)


for p in [me, opp]:

    print("\n", p.name)

    for stat in [
        "hp",
        "atk",
        "def",
        "spa",
        "spd",
        "spe"
    ]:

        try:
            print(
                stat,
                "=>",
                get_modified_stat(p, stat)
            )

        except Exception as e:
            print(
                stat,
                "ERROR",
                e
            )


print("="*60)

STAT CHECK

 Pikachu
hp => 100
atk => 120
def => 120
spa => 120
spd => 120
spe => 120

 Charizard
hp => 100
atk => 120
def => 120
spa => 120
spd => 120
spe => 120


In [ ]:
# ==========================================
# Move Database Check
# ==========================================

from engine.exact_mechanics import get_move


print("="*60)
print("MOVE CHECK")
print("="*60)


for m in me.moves:

    name = getattr(m, "name", str(m))

    result = get_move(name)

    print(
        name,
        "=>",
        result
    )


print("="*60)

MOVE CHECK
かえんほうしゃ => None
10まんボルト => None
れいとうビーム => None
まもる => None


In [ ]:
# ==========================================
# get_move source check
# ==========================================

import inspect
from engine.exact_mechanics import get_move

print("="*60)
print(inspect.getsource(get_move))
print("="*60)

def get_move(name: str) -> Move | None:
    key = to_id(name)
    return MOVE_DATABASE.get(key)



In [ ]:
# ==========================================
# Move Key Debug
# ==========================================

from data.move_database import MOVE_DATABASE
from data.move_database import to_id


print("="*60)
print("KEY DEBUG")
print("="*60)


for name in [
    "かえんほうしゃ",
    "10まんボルト",
    "れいとうビーム",
    "まもる"
]:

    key = to_id(name)

    print(
        name,
        "=>",
        key,
        "exists:",
        key in MOVE_DATABASE
    )


print("="*60)

print("DB sample keys:")

for k in list(MOVE_DATABASE.keys())[:30]:
    print(k)

KEY DEBUG
かえんほうしゃ => かえんほうしゃ exists: False
10まんボルト => 10まんボルト exists: False
れいとうビーム => れいとうビーム exists: False
まもる => まもる exists: False
DB sample keys:
tackle
quick-attack
flamethrower
fire-blast
surf
hydro-pump
thunderbolt
earthquake
leaf-blade
ice-beam
dragon-claw
shadow-ball
sludge-bomb
iron-head
body-press
protect
recover
swords-dance
dragon-dance
calm-mind
tailwind
will-o-wisp
moonblast
make-it-rain
knock-off


In [ ]:
# ==========================================
# Add Japanese Move Name Support
# ==========================================

path = "/content/engine/exact_mechanics.py"

with open(path, "r") as f:
    code = f.read()


old = """
def get_move(name: str) -> Move | None:
    key = to_id(name)
    return MOVE_DATABASE.get(key)
"""


new = """
def get_move(name: str) -> Move | None:

    key = to_id(name)

    move = MOVE_DATABASE.get(key)

    if move is not None:
        return move


    JP_MOVE_MAP = {
        "かえんほうしゃ": "flamethrower",
        "10まんボルト": "thunderbolt",
        "れいとうビーム": "ice-beam",
        "まもる": "protect",
        "じこさいせい": "recover",
        "つるぎのまい": "swords-dance",
        "りゅうのまい": "dragon-dance",
        "めいそう": "calm-mind",
        "おいかぜ": "tailwind",
        "おにび": "will-o-wisp",
    }


    jp_key = JP_MOVE_MAP.get(name)

    if jp_key:
        return MOVE_DATABASE.get(jp_key)


    return None
"""


if old not in code:
    print("target not found")
else:
    code = code.replace(old, new)

    with open(path, "w") as f:
        f.write(code)

    print("Japanese move mapping added")

target not found


In [ ]:
from engine.exact_mechanics import get_move

for x in [
    "かえんほうしゃ",
    "10まんボルト",
    "れいとうビーム",
    "まもる"
]:
    print(x, "=>", get_move(x))

かえんほうしゃ => None
10まんボルト => None
れいとうビーム => None
まもる => None


In [ ]:
# ==========================================
# get_move current source check
# ==========================================

import inspect
from engine.exact_mechanics import get_move

print(inspect.getsource(get_move))

def get_move(name: str) -> Move | None:
    key = to_id(name)
    return MOVE_DATABASE.get(key)



In [ ]:
# ==========================================
# Force patch get_move
# ==========================================

import engine.exact_mechanics as em
from data.move_database import MOVE_DATABASE


JP_MOVE_MAP = {
    "かえんほうしゃ": "flamethrower",
    "10まんボルト": "thunderbolt",
    "れいとうビーム": "ice-beam",
    "まもる": "protect",
    "じこさいせい": "recover",
    "つるぎのまい": "swords-dance",
    "りゅうのまい": "dragon-dance",
    "めいそう": "calm-mind",
    "おいかぜ": "tailwind",
    "おにび": "will-o-wisp",
}


def fixed_get_move(name):

    if name in JP_MOVE_MAP:
        return MOVE_DATABASE.get(JP_MOVE_MAP[name])

    key = em.to_id(name)

    return MOVE_DATABASE.get(key)


em.get_move = fixed_get_move


print("get_move patched")

get_move patched


In [ ]:
for x in [
    "かえんほうしゃ",
    "10まんボルト",
    "れいとうビーム",
    "まもる"
]:
    print(x, "=>", em.get_move(x))

かえんほうしゃ => Move(name='flamethrower', type='fire', category='special', power=90, accuracy=100, priority=0, pp=10)
10まんボルト => Move(name='thunderbolt', type='electric', category='special', power=90, accuracy=100, priority=0, pp=10)
れいとうビーム => Move(name='ice-beam', type='ice', category='special', power=90, accuracy=100, priority=0, pp=10)
まもる => Move(name='protect', type='normal', category='status', power=0, accuracy=100, priority=4, pp=10)


In [ ]:
# ==========================================
# Damage Test After Move Fix
# ==========================================

from engine.exact_mechanics import damage_rolls


print("="*60)
print("Damage Test After Move Fix")
print("="*60)


for m in me.moves:

    name = getattr(m, "name", str(m))

    result = damage_rolls(
        me,
        opp,
        name,
        real_state.field
    )

    print(
        name,
        "damage:",
        result.avg_damage,
        "KO:",
        result.ko_chance
    )


print("="*60)

Damage Test After Move Fix


AttributeError: 'MockPokemon' object has no attribute 'boosts'

In [ ]:
# ==========================================
# MockPokemon boosts compatibility
# ==========================================

me.boosts = {
    "atk": 0,
    "def": 0,
    "spa": 0,
    "spd": 0,
    "spe": 0,
}

opp.boosts = {
    "atk": 0,
    "def": 0,
    "spa": 0,
    "spd": 0,
    "spe": 0,
}

print("boosts added")

boosts added


In [ ]:
from engine.exact_mechanics import damage_rolls

for m in me.moves:

    name = getattr(m, "name", str(m))

    result = damage_rolls(
        me,
        opp,
        name,
        real_state.field
    )

    print(
        name,
        "damage:",
        result.avg_damage,
        "KO:",
        result.ko_chance
    )

かえんほうしゃ damage: 38.0 KO: 0.0
10まんボルト damage: 38.0 KO: 0.0
れいとうビーム damage: 38.0 KO: 0.0
まもる damage: 0.0 KO: 0.0


In [ ]:
# ==========================================
# Type Multiplier Check
# ==========================================

from data.type_chart import type_multiplier


print("="*60)
print("TYPE CHECK")
print("="*60)


for t in [
    "fire",
    "electric",
    "ice"
]:

    result = type_multiplier(
        t,
        getattr(opp, "types", ())
    )

    print(
        t,
        "=>",
        result
    )


print("Opponent types:")
print(opp.types)

print("="*60)

TYPE CHECK
fire => 1.0
electric => 1.0
ice => 1.0
Opponent types:
['normal']


In [ ]:
# ==========================================
# Fix Pokemon Types
# ==========================================

opp.types = [
    "fire",
    "flying"
]

me.types = [
    "electric"
]

print("Types fixed")

print(
    me.name,
    me.types
)

print(
    opp.name,
    opp.types
)

Types fixed
Pikachu ['electric']
Charizard ['fire', 'flying']


In [ ]:
from data.type_chart import type_multiplier

for t in [
    "fire",
    "electric",
    "ice"
]:
    print(
        t,
        "=>",
        type_multiplier(
            t,
            opp.types
        )
    )

fire => 0.5
electric => 2.0
ice => 1.0


In [ ]:
# ==========================================
# Action Value Real Test v2
# ==========================================

from engine.action_value_v1 import action_value
from battle.action import Action, ACTION_MOVE


print("="*60)
print("Action Value Test v2")
print("="*60)


for move in real_state.player.moves:

    name = getattr(move, "name", str(move))

    action = Action(
        action_type=ACTION_MOVE,
        move=name
    )

    score = action_value(
        real_state,
        action
    )

    print(
        name,
        "=>",
        score
    )


print("="*60)

Action Value Test v2
かえんほうしゃ => 41.112500000000004
10まんボルト => 19753.0
れいとうビーム => 83.60000000000001
まもる => 0.0


In [ ]:
# ==========================================
# MCTS + Action Value Integration Test
# ==========================================

from ai.mcts import MCTS
from engine.action_value_v1 import action_value
from battle.action import Action, ACTION_MOVE


actions = []

for move in real_state.player.moves:

    actions.append(
        Action(
            action_type=ACTION_MOVE,
            move=getattr(move,"name",str(move))
        )
    )


def rollout_fn(state, action):

    if action is None:
        return 0

    return action_value(
        state,
        action
    )


mcts = MCTS(
    iterations=100
)


result = mcts.search(
    real_state,
    actions,
    rollout_fn
)


print("="*60)
print("MCTS RESULT")
print("="*60)

print(result)

print("="*60)

MCTS RESULT
Action(action_type='move', move='10まんボルト', switch_index=None)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from copy import deepcopy
import math
import random


@dataclass
class Node:

    state: object

    parent: "Node | None" = None

    action: object | None = None

    children: list["Node"] = field(default_factory=list)

    visits: int = 0

    value: float = 0.0

    untried_actions: list = field(default_factory=list)



def uct_score(parent_visits, child):

    if child.visits == 0:
        return float("inf")

    exploit = child.value / child.visits

    explore = 1.414 * math.sqrt(
        math.log(parent_visits + 1)
        /
        child.visits
    )

    return exploit + explore



def evaluate_state(state):

    """
    MCTS rollout評価
    """

    score = 0


    try:

        me = state.player

        opp = state.opponent


        # HP評価

        my_hp = (
            me.current_hp /
            max(1, me.max_hp)
        )

        opp_hp = (
            opp.current_hp /
            max(1, opp.max_hp)
        )


        score += my_hp * 300

        score -= opp_hp * 300



        # KO評価

        if opp.current_hp <= 0:
            score += 10000


        if me.current_hp <= 0:
            score -= 10000



        # 素早さ評価

        if hasattr(me, "base_stats"):

            my_speed = me.base_stats.get(
                "spe",
                0
            )

            opp_speed = opp.base_stats.get(
                "spe",
                0
            )

            score += (
                my_speed -
                opp_speed
            )


    except Exception:

        pass


    return score



def mcts_choose_action(
    state,
    actions,
    rollout_fn,
    iterations=200
):


    root = Node(
        state=deepcopy(state),
        untried_actions=list(actions)
    )


    if not actions:
        return None



    for _ in range(iterations):


        node = root


        # Selection

        while (
            node.children
            and
            not node.untried_actions
        ):

            node = max(
                node.children,
                key=lambda x:
                uct_score(
                    node.visits,
                    x
                )
            )



        # Expansion

        if node.untried_actions:


            action = (
                node.untried_actions.pop()
            )


            child_state = deepcopy(
                node.state
            )


            reward = rollout_fn(
                child_state,
                action
            )


            reward += evaluate_state(
                child_state
            )


            child = Node(
                state=child_state,
                parent=node,
                action=action,
                visits=1,
                value=reward
            )


            node.children.append(
                child
            )


        else:


            reward = rollout_fn(
                deepcopy(node.state),
                None
            )


            reward += evaluate_state(
                node.state
            )



        # Backpropagation

        while node:

            node.visits += 1

            node.value += reward

            node = node.parent



    best = max(
        root.children,
        key=lambda x:x.visits,
        default=None
    )


    if best:

        return best.action


    return random.choice(actions)



class MCTS:


    def __init__(self, iterations=200):

        self.iterations = iterations



    def search(
        self,
        state,
        actions,
        rollout_fn
    ):

        return mcts_choose_action(
            state,
            actions,
            rollout_fn,
            self.iterations
        )

In [ ]:
import importlib
import ai.mcts

importlib.reload(ai.mcts)

print("MCTS v2 reload OK")

MCTS v2 reload OK


In [ ]:
from ai.mcts import MCTS

mcts = MCTS(iterations=200)

print(type(mcts))

<class 'ai.mcts.MCTS'>


In [ ]:
# ==========================================
# MCTS v2 Battle Decision Test
# ==========================================

from ai.mcts import MCTS
from engine.action_value_v1 import action_value
from battle.action import Action, ACTION_MOVE


actions = []

for move in real_state.player.moves:

    actions.append(
        Action(
            action_type=ACTION_MOVE,
            move=getattr(move, "name", str(move))
        )
    )


def rollout_fn(state, action):

    if action is None:
        return 0

    return action_value(
        state,
        action
    )


mcts = MCTS(
    iterations=200
)


result = mcts.search(
    real_state,
    actions,
    rollout_fn
)


print("="*60)
print("MCTS v2 RESULT")
print("="*60)

print(result)

print("="*60)

MCTS v2 RESULT
Action(action_type='move', move='10まんボルト', switch_index=None)


In [ ]:
# ==========================================
# Create battle_loop.py
# ==========================================

path = "/content/battle/battle_loop.py"


code = r'''
from ai.battle_ai import BattleAI
from engine.exact_mechanics import damage_rolls


class BattleLoop:


    def __init__(
        self,
        state,
        player_ai=None,
        opponent_ai=None
    ):

        self.state = state

        self.player_ai = (
            player_ai
            or
            BattleAI()
        )

        self.opponent_ai = (
            opponent_ai
            or
            BattleAI()
        )

        self.turn = 1



    def execute_move(
        self,
        attacker,
        defender,
        move
    ):

        result = damage_rolls(
            attacker,
            defender,
            move,
            self.state.field
        )


        damage = int(
            result.avg_damage
        )


        defender.current_hp = max(
            0,
            defender.current_hp - damage
        )


        return damage



    def step(self):


        print(
            "====== TURN",
            self.turn,
            "======"
        )


        player = self.state.player

        opponent = self.state.opponent



        # Player AI

        player_action = (
            self.player_ai.decide(
                self.state,
                player,
                opponent
            )
        )


        # Opponent AI

        opponent_action = (
            self.opponent_ai.decide(
                self.state,
                opponent,
                player
            )
        )



        print(
            "PLAYER:",
            player_action
        )


        print(
            "OPPONENT:",
            opponent_action
        )



        # Player move

        if player_action["type"] == "move":

            damage = self.execute_move(
                player,
                opponent,
                player_action["move"]
            )

            print(
                player.name,
                "damage",
                damage
            )



        if opponent.current_hp <= 0:

            return "PLAYER WIN"



        # Opponent move

        if opponent_action["type"] == "move":

            damage = self.execute_move(
                opponent,
                player,
                opponent_action["move"]
            )

            print(
                opponent.name,
                "damage",
                damage
            )



        if player.current_hp <= 0:

            return "OPPONENT WIN"



        self.turn += 1


        return "CONTINUE"



    def run(
        self,
        max_turns=50
    ):


        for _ in range(max_turns):

            result = self.step()


            print(
                "HP:",
                self.state.player.current_hp,
                "/",
                self.state.opponent.current_hp
            )


            if result != "CONTINUE":

                print(
                    result
                )

                return result


        return "DRAW"
'''


with open(path,"w") as f:
    f.write(code)


print("battle_loop.py created")

battle_loop.py created


In [ ]:
import importlib
import battle.battle_loop

importlib.reload(battle.battle_loop)

print("BattleLoop loaded")

BattleLoop loaded


In [ ]:
from battle.battle_loop import BattleLoop


# HP初期化

real_state.player.current_hp = real_state.player.max_hp
real_state.opponent.current_hp = real_state.opponent.max_hp


battle = BattleLoop(real_state)


result = battle.run(
    max_turns=10
)


print("RESULT:", result)

====== TURN 1 ======
PLAYER: {'type': 'move', 'move': '10まんボルト'}
OPPONENT: {'type': 'move', 'move': None}
Pikachu damage 115
HP: 100 / 0
PLAYER WIN
RESULT: PLAYER WIN


In [ ]:
# ==========================================
# BattleAI v2
# ==========================================

path = "/content/ai/battle_ai.py"


code = r'''
from ai.move_selector import MoveSelector
from battle.action import Action, ACTION_MOVE


class BattleAI:


    def __init__(self):

        self.selector = MoveSelector()



    def decide(
        self,
        state,
        me,
        opponent
    ):


        moves = getattr(
            me,
            "moves",
            []
        )


        if not moves:

            return {
                "type":"move",
                "move":None
            }



        move = self.selector.choose_move(
            state,
            me,
            opponent,
            moves
        )


        # Action形式対応

        if isinstance(move, Action):

            return {
                "type":move.action_type,
                "move":move.move
            }



        # Move object

        name = getattr(
            move,
            "name",
            move
        )


        return {
            "type":ACTION_MOVE,
            "move":name
        }
'''


with open(path,"w") as f:
    f.write(code)


print("BattleAI v2 written")

BattleAI v2 written


In [ ]:
import importlib
import ai.battle_ai

importlib.reload(ai.battle_ai)

print("BattleAI v2 reload OK")

BattleAI v2 reload OK


In [ ]:
from ai.battle_ai import BattleAI


ai = BattleAI()


print(
    "PLAYER:",
    ai.decide(
        real_state,
        real_state.player,
        real_state.opponent
    )
)


print(
    "OPPONENT:",
    ai.decide(
        real_state,
        real_state.opponent,
        real_state.player
    )
)

PLAYER: {'type': 'move', 'move': 'かえんほうしゃ'}
OPPONENT: {'type': 'move', 'move': None}


In [ ]:
# ==========================================
# MoveSelector Check
# ==========================================

import inspect
from ai.move_selector import MoveSelector


print(inspect.getsource(MoveSelector.choose_move))

    def choose_move(self, state, pokemon, opponent, moves):

        best_move = None
        best_score = -999999

        for move in moves:

            score = self.simulate(
                state,
                pokemon,
                opponent,
                move
            )

            if score > best_score:
                best_score = score
                best_move = move

        return best_move



In [ ]:
print("="*60)

print("PLAYER MOVES")

for m in real_state.player.moves:
    print(getattr(m,"name",m))


print("OPPONENT MOVES")

for m in real_state.opponent.moves:
    print(getattr(m,"name",m))

print("="*60)

PLAYER MOVES
かえんほうしゃ
10まんボルト
れいとうビーム
まもる
OPPONENT MOVES


In [ ]:
# ==========================================
# Add Opponent Moves
# ==========================================

opp.moves = [
    "かえんほうしゃ",
    "エアスラッシュ",
    "りゅうのはどう",
    "まもる"
]


print("Opponent moves added")


for m in opp.moves:
    print(m)

Opponent moves added
かえんほうしゃ
エアスラッシュ
りゅうのはどう
まもる


In [ ]:
opp.moves = [
    "flamethrower",
    "air-slash",
    "dragon-pulse",
    "protect"
]

In [ ]:
print("OPPONENT MOVES")

for m in opp.moves:
    print(m)

OPPONENT MOVES
flamethrower
air-slash
dragon-pulse
protect


In [132]:
print(
    ai.decide(
        real_state,
        real_state.opponent,
        real_state.player
    )
)

{'type': 'move', 'move': 'flamethrower'}
